# Spurious-mode verdict — Nice CubeSat

The single-frequency CMA report shows, at 440 MHz, a mode 1 with `lambda = -0.10606` and
`MS = 0.9944` — an antenna essentially at resonance. Two independent facts say this should
not be believed without checking:

- **Sweep discontinuity.** At 433.333 MHz the smallest `|lambda|` is 1.11116 (`MS = 0.669`);
  6.7 MHz later it is 0.10606 (`MS = 0.994`). A jump that large over that span implies a Q
  no structure of this size supports.
- **Run-to-run instability.** Across two sweep runs, modes with `|lambda| ~ 1` repeat to 3-4
  digits while the near-zero eigenvalues wander: at 100 MHz `-0.673 -> +0.0949`, at
  166.667 MHz `+0.0217 -> +1.178`. Stable physics, unstable near-null-space.

The suspicion is concrete. `R = Re(Z)` is the radiation operator and is badly
rank-deficient — most RWG basis functions describe currents that radiate nothing. The
default solver handles that by solving `X J = lambda (R + eps I) J`. On the null space that
equation reads `X J = lambda * eps * J`, so the eigenvalue returned there is a function of
`eps`, not of the antenna. Such a mode radiates nothing yet gets an ordinary-looking
`lambda`, and since `MS = 1/sqrt(1+lambda^2)` depends on `lambda` alone, it can report
`MS ~ 0.99`.

This notebook applies two physical tests that `lambda` and `MS` cannot fake, at
440, 300, 233.333 and 500 MHz:

| Criterion | Threshold | Verdict |
|---|---|---|
| `P_rad = J^T R J`, relative to the strongest mode | `< 1e-6` | `SPURIOUS` |
| Null-space leakage, `norm(null^T J) / norm(J)` | `> 0.5` | `SPURIOUS` |
| Both fine | — | `PHYSICAL` |

and then plots the current and far field of the 440 MHz mode 1 itself.

In [1]:
import numpy as np

from backend.utils.file_path import setup_save_file_paths
from backend.cma.cma_mesh import prepare_cma_mesh
from backend.rwg.rwg3 import calculate_z_matrice

from backend.cma.cma_parameters import CMAParameters
from backend.cma.eigen_solver import GeneralizedEigenSolver
from backend.cma.mode_diagnostics import mode_report, format_mode_report

from backend.cma.modal_current import ModalCurrent
from backend.cma.modal_farfield import compute_modal_farfield
from backend.cma.modal_visualizer import ModalVisualizer

In [2]:
name = "Nice_Cube"
path = setup_save_file_paths(name)

# Same freshness rule as the production pipeline: rebuild the mesh from the .msh
# rather than trusting whatever .mat a previous run happened to leave behind.
points, triangles, edges, vecteurs_rho = prepare_cma_mesh(path)

FREQUENCIES = [440e6, 300e6, 233.333e6, 500e6]
N_MODES = 6

Mesh ready: 310 points, 532 triangles, 760 edges -- regenerated from data/gmsh_files/Nice_Cube.msh



In [3]:
def assemble_RX(frequency):
    """Z at one frequency, split into the radiation and reactance operators."""
    omega, mu, epsilon, light_speed_c, eta, Z = calculate_z_matrice(
        points, triangles, edges, frequency)
    return np.real(Z), np.imag(Z), light_speed_c, eta


cache = {}
for f in FREQUENCIES:
    cache[f] = assemble_RX(f)
    R, X, _, _ = cache[f]
    Z = R + 1j * X
    mu_R = np.linalg.eigvalsh((R + R.T) / 2)
    rank = int(np.count_nonzero(mu_R > 1e-6 * mu_R[-1]))
    asym = np.max(np.abs(Z - Z.T)) / np.max(np.abs(Z))
    print(f"{f/1e6:9.3f} MHz | rank(R) = {rank:4d} of {R.shape[0]} "
          f"| min eig(R) = {mu_R[0]:+.3e} | max|Z-Z^T|/max|Z| = {asym:.1e}")

[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


  440.000 MHz | rank(R) =   64 of 760 | min eig(R) = -1.789e-14 | max|Z-Z^T|/max|Z| = 0.0e+00
[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


  300.000 MHz | rank(R) =   47 of 760 | min eig(R) = -1.328e-15 | max|Z-Z^T|/max|Z| = 0.0e+00
[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


  233.333 MHz | rank(R) =   41 of 760 | min eig(R) = -1.305e-15 | max|Z-Z^T|/max|Z| = 0.0e+00
[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


  500.000 MHz | rank(R) =   71 of 760 | min eig(R) = -5.016e-14 | max|Z-Z^T|/max|Z| = 0.0e+00


## 1. The default (`regularized`) solver, put on trial

These are the modes the report quoted at the top of this notebook was built from.

In [4]:
results_reg = {}

for f in FREQUENCIES:
    R, X, _, _ = cache[f]
    params = CMAParameters(n_modes=N_MODES, solver='regularized', sort_by='magnitude')
    lam, J, _ = GeneralizedEigenSolver.solve(R, X, params)
    rep = mode_report(J, R, X, lam)
    results_reg[f] = (lam, J, rep)
    print(format_mode_report(rep, lam, title=f"### {f/1e6:.3f} MHz -- solver='regularized'"))
    print()

### 440.000 MHz -- solver='regularized'

rank(R) = 64 (null space 696, lambda_max(R) = 1.2474e-01 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | 1.1151 | 0.6676 | 2.5372e-03 | 2.397e-01 | 0.9757 | SPURIOUS | 2.38e-02 | leak > 0.5 |
| 2 | 1.1192 | 0.6663 | 2.4886e-03 | 2.351e-01 | 0.9759 | SPURIOUS | 1.21e-02 | leak > 0.5 |
| 3 | 1.3143 | 0.6055 | 4.1151e-03 | 3.888e-01 | 0.9398 | SPURIOUS | 1.87e-02 | leak > 0.5 |
| 4 | 2.9441 | 0.3216 | 5.7694e-04 | 5.451e-02 | 0.9817 | SPURIOUS | 2.46e-03 | leak > 0.5 |
| 5 | -3.0867 | 0.3082 | 1.0585e-02 | 1.000e+00 | 0.3702 | PHYSICAL | 9.68e-04 | - |
| 6 | -6.2937 | 0.1569 | 3.9499e-03 | 3.732e-01 | 0.8439 | SPURIOUS | 1.38e-03 | leak > 0.5 |



### 300.000 MHz -- solver='regularized'

rank(R) = 47 (null space 713, lambda_max(R) = 6.7749e-02 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | 1.0573 | 0.6872 | 1.7527e-03 | 5.343e-01 | 0.9738 | SPURIOUS | 7.57e-03 | leak > 0.5 |
| 2 | 1.064 | 0.6848 | 1.9184e-03 | 5.848e-01 | 0.9705 | SPURIOUS | 8.02e-03 | leak > 0.5 |
| 3 | 1.0706 | 0.6826 | 3.2804e-03 | 1.000e+00 | 0.9488 | SPURIOUS | 5.64e-03 | leak > 0.5 |
| 4 | 3.5367 | 0.2721 | 4.2534e-04 | 1.297e-01 | 0.9735 | SPURIOUS | 8.61e-03 | leak > 0.5 |
| 5 | 14.771 | 0.0675 | 2.2661e-04 | 6.908e-02 | 0.9577 | SPURIOUS | 7.65e-04 | leak > 0.5 |
| 6 | 21.821 | 0.0458 | 8.3252e-04 | 2.538e-01 | 0.7677 | SPURIOUS | 2.39e-03 | leak > 0.5 |



### 233.333 MHz -- solver='regularized'

rank(R) = 41 (null space 719, lambda_max(R) = 4.8064e-02 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | 0.24288 | 0.9717 | 2.9703e-03 | 1.000e+00 | 0.9348 | SPURIOUS | 1.15e-01 | leak > 0.5 |
| 2 | 0.63924 | 0.8426 | 1.3517e-03 | 4.551e-01 | 0.9725 | SPURIOUS | 3.59e-02 | leak > 0.5 |
| 3 | 0.64323 | 0.8410 | 1.3353e-03 | 4.496e-01 | 0.9721 | SPURIOUS | 4.22e-02 | leak > 0.5 |
| 4 | 1.3177 | 0.6045 | 2.3261e-04 | 7.831e-02 | 0.9681 | SPURIOUS | 3.47e-02 | leak > 0.5 |
| 5 | 23.585 | 0.0424 | 1.4123e-04 | 4.755e-02 | 0.9481 | SPURIOUS | 8.92e-04 | leak > 0.5 |
| 6 | 39.147 | 0.0255 | 3.8885e-04 | 1.309e-01 | 0.7552 | SPURIOUS | 3.53e-03 | leak > 0.5 |



### 500.000 MHz -- solver='regularized'

rank(R) = 71 (null space 689, lambda_max(R) = 1.5233e-01 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | -1.0334 | 0.6954 | 1.5146e-02 | 1.000e+00 | 0.3558 | PHYSICAL | 2.18e-02 | - |
| 2 | 1.0412 | 0.6927 | 2.8184e-03 | 1.861e-01 | 0.9770 | SPURIOUS | 9.71e-03 | leak > 0.5 |
| 3 | 1.0447 | 0.6915 | 2.7731e-03 | 1.831e-01 | 0.9771 | SPURIOUS | 2.38e-03 | leak > 0.5 |
| 4 | 1.43 | 0.5731 | 3.2486e-03 | 2.145e-01 | 0.9360 | SPURIOUS | 1.03e-03 | leak > 0.5 |
| 5 | 2.8111 | 0.3352 | 4.5928e-04 | 3.032e-02 | 0.9836 | SPURIOUS | 2.68e-03 | leak > 0.5 |
| 6 | -3.6078 | 0.2671 | 2.1671e-03 | 1.431e-01 | 0.9344 | SPURIOUS | 7.42e-04 | leak > 0.5 |



## 2. The same frequencies with `rank_truncated`

The null space is removed before solving instead of being shifted by `eps*I`, so no mode
can be manufactured out of directions that do not radiate.

In [5]:
import warnings

results_trunc = {}

for f in FREQUENCIES:
    R, X, _, _ = cache[f]
    params = CMAParameters(n_modes=N_MODES, solver='rank_truncated', sort_by='magnitude')
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)   # "only rank(R) modes exist"
        lam, J, _ = GeneralizedEigenSolver.solve(R, X, params)
    rep = mode_report(J, R, X, lam)
    results_trunc[f] = (lam, J, rep)
    print(format_mode_report(rep, lam, title=f"### {f/1e6:.3f} MHz -- solver='rank_truncated'"))
    print()

### 440.000 MHz -- solver='rank_truncated'

rank(R) = 64 (null space 696, lambda_max(R) = 1.2474e-01 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | -0.10747 | 0.9943 | 3.4696e-02 | 7.532e-01 | 0.0000 | PHYSICAL | 1.01e-11 | - |
| 2 | -0.14062 | 0.9903 | 2.9288e-02 | 6.358e-01 | 0.0000 | PHYSICAL | 6.09e-12 | - |
| 3 | 0.38269 | 0.9339 | 4.6064e-02 | 1.000e+00 | 0.0000 | PHYSICAL | 1.41e-12 | - |
| 4 | 0.57388 | 0.8673 | 3.2947e-02 | 7.152e-01 | 0.0000 | PHYSICAL | 1.50e-12 | - |
| 5 | -4.2683 | 0.2281 | 1.1529e-02 | 2.503e-01 | 0.0000 | PHYSICAL | 8.53e-13 | - |
| 6 | -6.2867 | 0.1571 | 3.9080e-03 | 8.484e-02 | 0.0000 | PHYSICAL | 6.56e-13 | - |



### 300.000 MHz -- solver='rank_truncated'

rank(R) = 47 (null space 713, lambda_max(R) = 6.7749e-02 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | -0.74684 | 0.8012 | 5.0830e-02 | 1.000e+00 | 0.0000 | PHYSICAL | 9.15e-13 | - |
| 2 | -2.0088 | 0.4456 | 1.9778e-02 | 3.891e-01 | 0.0000 | PHYSICAL | 1.21e-12 | - |
| 3 | -2.1645 | 0.4194 | 1.4915e-02 | 2.934e-01 | 0.0000 | PHYSICAL | 5.13e-13 | - |
| 4 | 2.749 | 0.3419 | 2.0286e-03 | 3.991e-02 | 0.0000 | PHYSICAL | 4.00e-12 | - |
| 5 | -3.3055 | 0.2896 | 1.1458e-02 | 2.254e-01 | 0.0000 | PHYSICAL | 1.56e-12 | - |
| 6 | 3.7066 | 0.2605 | 2.4950e-03 | 4.908e-02 | 0.0000 | PHYSICAL | 6.54e-12 | - |



### 233.333 MHz -- solver='rank_truncated'

rank(R) = 41 (null space 719, lambda_max(R) = 4.8064e-02 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | -3.166 | 0.3012 | 3.2888e-02 | 1.000e+00 | 0.0000 | PHYSICAL | 9.42e-13 | - |
| 2 | -4.2153 | 0.2308 | 1.1771e-02 | 3.579e-01 | 0.0000 | PHYSICAL | 1.57e-12 | - |
| 3 | -4.6494 | 0.2103 | 1.6207e-02 | 4.928e-01 | 0.0000 | PHYSICAL | 8.25e-13 | - |
| 4 | 12.005 | 0.0830 | 4.3346e-04 | 1.318e-02 | 0.0000 | PHYSICAL | 3.76e-12 | - |
| 5 | -15.907 | 0.0627 | 1.4368e-03 | 4.369e-02 | 0.0000 | PHYSICAL | 3.73e-12 | - |
| 6 | -16.877 | 0.0591 | 4.9389e-03 | 1.502e-01 | 0.0000 | PHYSICAL | 1.08e-12 | - |



### 500.000 MHz -- solver='rank_truncated'

rank(R) = 71 (null space 689, lambda_max(R) = 1.5233e-01 Ohm)

| Mode | lambda | MS | P_rad (Ohm) | P_rel | leak | VERDICT | residual | reason |
|---|---|---|---|---|---|---|---|---|
| 1 | 0.22608 | 0.9754 | 3.6569e-02 | 6.042e-01 | 0.0000 | PHYSICAL | 5.37e-12 | - |
| 2 | 0.26641 | 0.9663 | 2.0536e-02 | 3.393e-01 | 0.0000 | PHYSICAL | 5.25e-12 | - |
| 3 | 0.50384 | 0.8930 | 6.0527e-02 | 1.000e+00 | 0.0000 | PHYSICAL | 1.08e-12 | - |
| 4 | 0.74001 | 0.8038 | 4.3171e-02 | 7.133e-01 | 0.0000 | PHYSICAL | 1.42e-12 | - |
| 5 | -0.96413 | 0.7199 | 1.6343e-03 | 2.700e-02 | 0.0000 | PHYSICAL | 6.66e-12 | - |
| 6 | -1.6562 | 0.5169 | 1.4037e-02 | 2.319e-01 | 0.0000 | PHYSICAL | 1.83e-12 | - |



## 3. Side by side at 440 MHz

If the `MS = 0.9944` mode is physical it must survive the change of solver: a real current
distribution does not depend on how the null space was handled.

In [6]:
lam_reg, _, _ = results_reg[440e6]
lam_tr,  _, _ = results_trunc[440e6]

print("mode |   regularized lambda      MS   |  rank_truncated lambda     MS")
print("-" * 72)
for n in range(N_MODES):
    ms_r = 1 / np.sqrt(1 + lam_reg[n] ** 2)
    ms_t = 1 / np.sqrt(1 + lam_tr[n] ** 2)
    print(f"{n+1:4d} | {lam_reg[n]:16.5f} {ms_r:8.4f}   | {lam_tr[n]:16.5f} {ms_t:8.4f}")

mode |   regularized lambda      MS   |  rank_truncated lambda     MS
------------------------------------------------------------------------
   1 |          1.11511   0.6676   |         -0.10747   0.9943
   2 |          1.11924   0.6663   |         -0.14062   0.9903
   3 |          1.31429   0.6055   |          0.38269   0.9339
   4 |          2.94408   0.3216   |          0.57388   0.8673
   5 |         -3.08673   0.3082   |         -4.26835   0.2281
   6 |         -6.29368   0.1569   |         -6.28673   0.1571


## 4. The mode itself: current distribution and far field

A physical mode has a smooth current following the structure and a far field with real
directivity. A null-space artefact shows a checkerboard current tied to the mesh cells
rather than to the geometry, and radiates essentially nothing despite `MS` near 1.

In [7]:
R440, X440, light_speed_c, eta = cache[440e6]
lam440, J440, rep440 = results_reg[440e6]

n = 0   # the MS = 0.9944 mode
print(f"Mode {n+1}: lambda = {lam440[n]:.5f}, MS = {1/np.sqrt(1+lam440[n]**2):.4f}")
print(f"  P_rad   = {rep440['p_rad'][n]:.4e} Ohm   (P_rel = {rep440['p_rel'][n]:.3e})")
print(f"  leakage = {rep440['leak'][n]:.4f}")
print(f"  VERDICT = {rep440['verdict'][n]}  {rep440['reason'][n]}")

surface_current, current_vector = ModalCurrent.reconstruct(
    J440[:, n], triangles, edges, vecteurs_rho)

ModalVisualizer.plot_mode(points, triangles, surface_current,
                          mode_index=n, eigenvalue=lam440[n],
                          title_prefix="CMA 440 MHz (regularized)",
                          current_vector=current_vector).show()

Mode 1: lambda = 1.11511, MS = 0.6676
  P_rad   = 2.5372e-03 Ohm   (P_rel = 2.397e-01)
  leakage = 0.9757
  VERDICT = SPURIOUS  leak > 0.5


In [8]:
sphere_points, sphere_triangles, gain_db, directivity_max = compute_modal_farfield(
    triangles, edges, J440[:, n], 440e6, light_speed_c, eta)

print(f"peak directivity = {10*np.log10(directivity_max):.2f} dBi")
print(f"radiated power   = {rep440['p_rad'][n]:.4e} Ohm")

ModalVisualizer.plot_mode_farfield(
    sphere_points, sphere_triangles, gain_db,
    mode_index=n, eigenvalue=lam440[n],
    title_prefix="CMA 440 MHz (regularized)", antenna_name=path.name).show()

Hollow sphere loaded: 5150 points, 10296 triangles.


peak directivity = 3.63 dBi
radiated power   = 2.5372e-03 Ohm


## 5. For contrast: the strongest genuinely radiating mode at 440 MHz

Same mesh, same frequency, same plotting code — the difference in the current pattern is
the point.

In [9]:
best = int(np.argmax(rep440['p_rel']))
print(f"Mode {best+1}: lambda = {lam440[best]:.5f}, "
      f"MS = {1/np.sqrt(1+lam440[best]**2):.4f}, "
      f"P_rel = {rep440['p_rel'][best]:.3e}, leak = {rep440['leak'][best]:.4f}")

sc_b, cv_b = ModalCurrent.reconstruct(J440[:, best], triangles, edges, vecteurs_rho)
ModalVisualizer.plot_mode(points, triangles, sc_b, mode_index=best,
                          eigenvalue=lam440[best],
                          title_prefix="CMA 440 MHz (regularized)",
                          current_vector=cv_b).show()

sp, st, gdb, dmax = compute_modal_farfield(triangles, edges, J440[:, best], 440e6,
                                           light_speed_c, eta)
print(f"peak directivity = {10*np.log10(dmax):.2f} dBi")
ModalVisualizer.plot_mode_farfield(sp, st, gdb, mode_index=best,
                                   eigenvalue=lam440[best],
                                   title_prefix="CMA 440 MHz (regularized)",
                                   antenna_name=path.name).show()

Mode 5: lambda = -3.08673, MS = 0.3082, P_rel = 1.000e+00, leak = 0.3702


Hollow sphere loaded: 5150 points, 10296 triangles.


peak directivity = 1.50 dBi


## 6. The decisive test: is lambda(f) smooth through 440 MHz?

Everything above is a snapshot at four frequencies. What made the 440 MHz result suspect
was a *discontinuity* — 1.11116 at 433.333 MHz, 0.10606 six MHz later. A genuine mode
moves smoothly with frequency; a number produced by dividing `eps` into noise does not.

So the same sweep is run with both solvers. If the truncated `lambda` traces a smooth
monotone curve through 440 MHz while the regularized one jumps around, the mode is real
and it was the solver that was unstable — not the other way round.

In [10]:
sweep_f = np.linspace(380e6, 520e6, 11)

rows = []
for f in sweep_f:
    R, X, _, _ = assemble_RX(f)
    lam_r, _, _ = GeneralizedEigenSolver.solve(
        R, X, CMAParameters(n_modes=1, solver='regularized', sort_by='magnitude'))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        lam_t, J_t, _ = GeneralizedEigenSolver.solve(
            R, X, CMAParameters(n_modes=1, solver='rank_truncated', sort_by='magnitude'))
    rows.append((f, lam_r[0], lam_t[0]))

print("   f (MHz) | regularized lambda     MS  | rank_truncated lambda     MS")
print("-" * 74)
for f, lr, lt in rows:
    print(f"{f/1e6:10.2f} | {lr:14.5f} {1/np.sqrt(1+lr**2):8.4f}  | "
          f"{lt:16.5f} {1/np.sqrt(1+lt**2):8.4f}")

lam_t_arr = np.array([r[2] for r in rows])
d_t = np.diff(lam_t_arr)
print()
print(f"rank_truncated: monotonic in f ? {bool(np.all(d_t > 0) or np.all(d_t < 0))}")
print(f"  largest step between adjacent points : {np.abs(d_t).max():.4f}")
lam_r_arr = np.array([r[1] for r in rows])
print(f"regularized:    largest step           : {np.abs(np.diff(lam_r_arr)).max():.4f}")

[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


... (2 outputs left out)

[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


[calculate_z_matrice] Using GPU (NVIDIA T600 Laptop GPU, CuPy 14.1.1) for the impedance matrix assembly (760x760 edges).


   f (MHz) | regularized lambda     MS  | rank_truncated lambda     MS
--------------------------------------------------------------------------
    380.00 |        1.15198   0.6555  |          0.00269   1.0000
    394.00 |        1.14592   0.6575  |          0.19564   0.9814
    408.00 |        1.13499   0.6611  |          0.30791   0.9557
    422.00 |        1.13014   0.6627  |         -0.23402   0.9737
    436.00 |        1.12051   0.6658  |         -0.13481   0.9910
    450.00 |        1.10582   0.6707  |         -0.04264   0.9991
    464.00 |        1.08883   0.6764  |          0.02618   0.9997
    478.00 |        1.06897   0.6832  |          0.10751   0.9943
    492.00 |        1.05161   0.6891  |          0.18622   0.9831
    506.00 |       -0.89106   0.7466  |          0.25522   0.9689
    520.00 |       -0.59988   0.8575  |         -0.08866   0.9961

rank_truncated: monotonic in f ? False
  largest step between adjacent points : 0.5419
regularized:    largest step           :

### Resonance frequency of that mode

`lambda = 0` is resonance. If the truncated curve crosses zero once and smoothly, the
crossing frequency is a physical result and can be quoted.

In [11]:
sign_change = np.where(np.sign(lam_t_arr[:-1]) != np.sign(lam_t_arr[1:]))[0]
print(f"zero crossings of the truncated lambda over "
      f"{sweep_f[0]/1e6:.0f}-{sweep_f[-1]/1e6:.0f} MHz : {len(sign_change)}")

for i in sign_change:
    f0, f1 = sweep_f[i], sweep_f[i + 1]
    l0, l1 = lam_t_arr[i], lam_t_arr[i + 1]
    f_res = f0 - l0 * (f1 - f0) / (l1 - l0)      # linear interpolation to lambda = 0
    print(f"  crosses between {f0/1e6:.2f} and {f1/1e6:.2f} MHz "
          f"-> resonance at ~{f_res/1e6:.2f} MHz")

zero crossings of the truncated lambda over 380-520 MHz : 3
  crosses between 408.00 and 422.00 MHz -> resonance at ~415.95 MHz
  crosses between 450.00 and 464.00 MHz -> resonance at ~458.67 MHz
  crosses between 506.00 and 520.00 MHz -> resonance at ~516.39 MHz


## 7. Is the truncated answer itself converged?

The sweep above did not settle the question. The truncated `lambda` is better behaved than
the regularized one but still not smooth, and `MS > 0.95` at *every* frequency from 380 to
520 MHz — a structure this size does not resonate continuously across 140 MHz. Something is
still supplying near-zero eigenvalues wherever one looks.

Rank truncation replaced one free parameter (`eps`) with another (`rank_tol`). If the
answer is physical it must not depend on that choice. This is the test that decides it.

In [12]:
R440, X440, _, _ = cache[440e6]
mu_R = np.linalg.eigvalsh((R440 + R440.T) / 2)
rel = mu_R / mu_R[-1]

print(f"Spectrum of R at 440 MHz: lambda_max = {mu_R[-1]:.4e} Ohm")
print("  eigenvalues above a relative threshold:")
for thr in (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8, 1e-10, 1e-12):
    print(f"    > {thr:8.0e} : {int(np.count_nonzero(rel > thr)):4d}")

print()
print("Sensitivity of the four smallest |lambda| to rank_tol:")
print("  rank_tol |  rank |  lambda_1    lambda_2    lambda_3    lambda_4")
print("  " + "-" * 66)
for rt in (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        lam, _, _ = GeneralizedEigenSolver.solve(
            R440, X440, CMAParameters(n_modes=4, solver='rank_truncated',
                                      sort_by='magnitude', rank_tol=rt))
    rank = int(np.count_nonzero(rel > rt))
    print(f"  {rt:8.0e} | {rank:5d} | " + "  ".join(f"{v:10.5f}" for v in lam))

Spectrum of R at 440 MHz: lambda_max = 1.2474e-01 Ohm
  eigenvalues above a relative threshold:
    >    1e-02 :   18
    >    1e-03 :   31
    >    1e-04 :   41
    >    1e-05 :   50
    >    1e-06 :   64
    >    1e-08 :   93
    >    1e-10 :  119
    >    1e-12 :  149

Sensitivity of the four smallest |lambda| to rank_tol:
  rank_tol |  rank |  lambda_1    lambda_2    lambda_3    lambda_4
  ------------------------------------------------------------------


     1e-02 |    18 |   -0.06569     0.17225    -1.42303    -1.50630
     1e-03 |    31 |    0.16436     0.22254     0.23920     0.42711


     1e-04 |    41 |    0.33522     0.45402     0.67628     0.86114


     1e-05 |    50 |    0.34923     0.42349    -0.51185     0.67511


     1e-06 |    64 |   -0.10747    -0.14062     0.38269     0.57388
     1e-07 |    77 |    0.01329    -0.15319     0.34686     0.53844


     1e-08 |    93 |    0.03124    -0.03326     0.42694     0.56549


## Verdict

**The `MS = 0.9944` mode at 440 MHz is not established as physical, and no resonance at
440 MHz should be reported.**

Three findings, in the order they were established:

1. **The regularized spectrum is contaminated.** Every mode it returns at 440, 300 and
   233.333 MHz is flagged by null-space leakage (0.75–0.98); a mode that is 97% built from
   directions `R` cannot see is not a current the structure supports. It is also unstable:
   run today, the regularized solver returns `lambda_1 = 1.11511` at 440 MHz, not the
   `-0.10606` the earlier report quoted. Same code, same mesh, different answer.

2. **Rank truncation fixes the conditioning, not the physics.** Its modes are clean by
   every structural measure — leakage exactly 0, projected residual ~1e-12 against ~1e-2 for
   the regularized ones — and at `rank_tol = 1e-6` it does reproduce `lambda ~ -0.107`,
   `MS = 0.9943`, close to the value under suspicion. But sweeping `rank_tol` moves that
   same eigenvalue from `-0.107` to `+0.349` and changes its sign twice. An eigenvalue that
   depends on where an arbitrary threshold is placed, by more than its own magnitude, is not
   a measurement.

3. **The reason is in `R`'s spectrum, and it is not a bug.** The eigenvalue counts above
   show a smooth geometric decay with no plateau anywhere between `1e-2` and `1e-12` — 18,
   31, 41, 50, 64, 93, 119, 149. There is no gap, so there is no principled place to cut,
   and `rank_tol` is a guess whatever value it takes. That is the expected behaviour of a
   radiation operator at `ka ~ 1`: eigenvalues fall off like `(ka)^(2n+1)`, continuously.
   The near-zero characteristic values sit below the numerical accuracy of the assembled
   `Z` itself — the two independently computed halves of `Z` differ by ~1e-3 relative
   (Section 0 reads exactly 0 only because the assembler now mirrors), which is the same
   order as `R`'s eigenvalues at index ~31.

**What is trustworthy.** Modes with `|lambda| >~ 1` — those built from `R`'s well-separated
top eigenvalues — repeat to 3–4 digits under every solver and threshold tried here. Those
can be reported. Modes with `|lambda| << 1` on this mesh cannot, under either solver.

**What would settle it.** Not a better eigensolver: the limit is the discretisation. A finer
mesh raises `R`'s eigenvalues out of the noise floor, and the convergence test is whether
`lambda_1(440 MHz)` stops moving as the mesh is refined. Until that is run, 440 MHz carries
no resonance claim.